In [1]:
import pandas as pd

events = pd.read_csv("events.csv")

# oldest first (stable sort keeps the original order for events with the same timestamp)
events = events.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

# clearer column names, matching our MySQL table
events = events.rename(columns={
    "timestamp": "event_ts", "visitorid": "visitor_id", "event": "event_type",
    "itemid": "item_id", "transactionid": "transaction_id"
})

events["transaction_id"] = events["transaction_id"].astype("Int64")   # whole numbers, not 123.0
events["event_id"]    = events.index + 1                              # 1, 2, 3 ... in time order
events["event_time"]  = pd.to_datetime(events["event_ts"], unit="ms") # readable time (UTC)
events["event_date"]  = events["event_time"].dt.date
events["event_hour"]  = events["event_time"].dt.hour
events["day_of_week"] = events["event_time"].dt.dayofweek             # 0 = Monday ... 6 = Sunday

events = events[["event_id", "event_ts", "event_time", "visitor_id", "event_type",
                 "item_id", "transaction_id", "event_date", "event_hour", "day_of_week"]]

print(events.shape)
events.head()

(2756101, 10)


,event_id,event_ts,event_time,visitor_id,event_type,item_id,transaction_id,event_date,event_hour,day_of_week
0,1,1430622004384,2015-05-03 03:00:04.384,693516,addtocart,297662,<NA>,2015-05-03,3,6
1,2,1430622011289,2015-05-03 03:00:11.289,829044,view,60987,<NA>,2015-05-03,3,6
2,3,1430622013048,2015-05-03 03:00:13.048,652699,view,252860,<NA>,2015-05-03,3,6
3,4,1430622024154,2015-05-03 03:00:24.154,1125936,view,33661,<NA>,2015-05-03,3,6
4,5,1430622026228,2015-05-03 03:00:26.228,693516,view,297662,<NA>,2015-05-03,3,6


In [2]:
n = len(events)
cutoff_ts = events["event_ts"].iloc[int(n * 0.6) - 1]      # timestamp at the 60% mark

baseline = events[events["event_ts"] <= cutoff_ts].copy()  # first 60% (same-timestamp events stay together)
replay   = events[events["event_ts"] >  cutoff_ts].copy()  # last 40%
baseline["batch_id"] = 0                                   # 0 = baseline

print("Cutoff time:", pd.to_datetime(cutoff_ts, unit="ms"))
print("Baseline:", len(baseline), f"({len(baseline)/n:.1%})")
print("Replay:  ", len(replay),   f"({len(replay)/n:.1%})")
print("Baseline + replay = total?", len(baseline) + len(replay) == n)

baseline.to_csv("events_baseline.csv", index=False)
replay.to_csv("events_replay.csv", index=False)

Cutoff time: 2015-07-21 20:32:23.468000
Baseline: 1653660 (60.0%)
Replay:   1102441 (40.0%)
Baseline + replay = total? True


In [3]:
ct = pd.read_csv("category_tree.csv")
parent = {int(c): (None if pd.isna(p) else int(p))
          for c, p in zip(ct["categoryid"], ct["parentid"])}

def walk_up(c):
    level, current, seen = 1, c, {c}
    while parent.get(current) is not None:
        current = parent[current]
        if current in seen:
            raise ValueError(f"Loop found at category {c}")
        seen.add(current)
        level += 1
    return level, current          # level, top-level category

rows = []
for c in parent:
    level, root = walk_up(c)
    rows.append((c, parent[c], level, root))

categories = pd.DataFrame(rows, columns=["category_id", "parent_id", "level", "root_category_id"])
categories["parent_id"] = categories["parent_id"].astype("Int64")
categories.to_csv("categories_prepared.csv", index=False)

print("Categories per level:", categories["level"].value_counts().sort_index().to_dict())
print("Top-level categories:", categories["root_category_id"].nunique())

Categories per level: {1: 25, 2: 174, 3: 702, 4: 665, 5: 90, 6: 13}
Top-level categories: 25


In [4]:
keep = []
for part in [1, 2]:
    for chunk in pd.read_csv(f"item_properties_part{part}.csv", chunksize=2_000_000, dtype={"value": str}):
        keep.append(chunk[chunk["property"].isin(["categoryid", "available"])])
    print("finished part", part)

props = pd.concat(keep, ignore_index=True)
props["valid_from"] = pd.to_datetime(props["timestamp"], unit="ms")
props = props.rename(columns={"itemid": "item_id"})

# --- category history
cat_hist = props[props["property"] == "categoryid"][["item_id", "valid_from", "value"]].copy()
cat_hist["category_id"] = pd.to_numeric(cat_hist["value"]).astype(int)
cat_hist = cat_hist.drop(columns="value").drop_duplicates().sort_values(["item_id", "valid_from"])
print("Conflicting category rows:", cat_hist.duplicated(["item_id", "valid_from"]).sum())
cat_hist = cat_hist.drop_duplicates(["item_id", "valid_from"], keep="last")

# --- availability history
av_hist = props[props["property"] == "available"][["item_id", "valid_from", "value"]].copy()
av_hist["available"] = pd.to_numeric(av_hist["value"]).astype(int)
av_hist = av_hist.drop(columns="value").drop_duplicates().sort_values(["item_id", "valid_from"])
print("Conflicting availability rows:", av_hist.duplicated(["item_id", "valid_from"]).sum())
av_hist = av_hist.drop_duplicates(["item_id", "valid_from"], keep="last")

# events before 10 May use the product's earliest known category:
# move each product's first category row back to 1 Jan 2015
first_idx = cat_hist.groupby("item_id")["valid_from"].idxmin()
cat_hist.loc[first_idx, "valid_from"] = pd.Timestamp("2015-01-01")

cat_hist.to_csv("item_category_history.csv", index=False)
av_hist.to_csv("item_availability_history.csv", index=False)

print("Category history rows:    ", len(cat_hist), "| items:", cat_hist["item_id"].nunique())
print("Availability history rows:", len(av_hist),  "| items:", av_hist["item_id"].nunique())

finished part 1
finished part 2
Conflicting category rows: 0
Conflicting availability rows: 0
Category history rows:     788214 | items: 417053
Availability history rows: 1503639 | items: 417053


In [5]:
all_items = pd.Series(
    pd.concat([events["item_id"], cat_hist["item_id"], av_hist["item_id"]]).unique(),
    name="item_id"
)
first_seen = baseline.groupby("item_id")["event_time"].min().rename("first_seen_time")

items = all_items.to_frame().merge(first_seen, on="item_id", how="left").sort_values("item_id")
items.to_csv("items_prepared.csv", index=False)

print("Products:", len(items))
print("Products with no baseline event (first_seen empty):", items["first_seen_time"].isna().sum())

Products: 466868
Products with no baseline event (first_seen empty): 277953


In [6]:
print("Category IDs missing from the tree:", (~cat_hist["category_id"].isin(categories["category_id"])).sum())
print("Baseline events whose product has a category: %.1f%%" % (baseline["item_id"].isin(cat_hist["item_id"]).mean() * 100))
print("Event IDs in both baseline and replay:", len(set(baseline["event_id"]) & set(replay["event_id"])))
print("All baseline events older than replay events?", baseline["event_ts"].max() < replay["event_ts"].min())
print("Baseline event types:", baseline["event_type"].value_counts().to_dict())
print("Replay event types:  ", replay["event_type"].value_counts().to_dict())

Category IDs missing from the tree: 167
Baseline events whose product has a category: 90.8%
Event IDs in both baseline and replay: 0
All baseline events older than replay events? True
Baseline event types: {'view': 1599672, 'addtocart': 40649, 'transaction': 13339}
Replay event types:   {'view': 1064640, 'addtocart': 28683, 'transaction': 9118}


In [1]:
import pandas as pd

replay = pd.read_csv("events_replay.csv", parse_dates=["event_time", "event_date"],
                     dtype={"transaction_id": "Int64"})
replay["event_date"] = replay["event_date"].dt.date

# the Monday of each event's week; the weeks are numbered 1, 2, 3 ... in time order
week_start = replay["event_time"].dt.to_period("W-SUN").dt.start_time.dt.normalize()
order = {w: i + 1 for i, w in enumerate(sorted(week_start.unique()))}
replay["batch_id"] = week_start.map(order).astype(int)
replay.to_csv("events_replay_batched.csv", index=False)

g = replay.assign(week_start=week_start).groupby("batch_id")
plan = g.agg(week_start=("week_start", "first"),
             batch_start=("event_time", "min"),
             batch_end=("event_time", "max"),
             events_planned=("event_id", "count")).reset_index()
plan["week_start"] = plan["week_start"].dt.date
for label, name in [("view", "views_planned"), ("addtocart", "carts_planned"), ("transaction", "purchases_planned")]:
    plan[name] = (replay[replay["event_type"] == label].groupby("batch_id").size()
                  .reindex(plan["batch_id"]).fillna(0).astype(int).values)
plan.to_csv("replay_plan.csv", index=False)
plan

,batch_id,week_start,batch_start,batch_end,events_planned,views_planned,carts_planned,purchases_planned
0,1,2015-07-20,2015-07-21 20:32:29.671,2015-07-26 23:59:58.790,131711,127712,3042,957
1,2,2015-07-27,2015-07-27 00:00:00.548,2015-08-02 23:59:55.653,144337,139112,3881,1344
2,3,2015-08-03,2015-08-03 00:00:01.588,2015-08-09 23:59:59.534,128559,123934,3552,1073
3,4,2015-08-10,2015-08-10 00:00:00.560,2015-08-16 23:59:55.927,120763,116721,3130,912
4,5,2015-08-17,2015-08-17 00:00:00.735,2015-08-23 23:59:56.475,126924,122430,3340,1154
5,6,2015-08-24,2015-08-24 00:00:12.490,2015-08-30 23:59:52.674,127810,123073,3603,1134
6,7,2015-08-31,2015-08-31 00:00:08.367,2015-09-06 23:59:58.684,119057,115227,2906,924
7,8,2015-09-07,2015-09-07 00:00:02.250,2015-09-13 23:59:56.743,138789,134142,3536,1111
8,9,2015-09-14,2015-09-14 00:00:05.345,2015-09-18 02:59:47.788,64491,62289,1693,509


In [2]:
base_max_id = pd.read_csv("events_baseline.csv", usecols=["event_id"])["event_id"].max()
r = pd.read_csv("events_replay_batched.csv", usecols=["event_id", "event_ts", "batch_id"])

n_batches = r["batch_id"].nunique()
print("Replay rows (expect 1102441):", len(r))
print("Batches:", n_batches, "| numbered 1..N without gaps:", sorted(r["batch_id"].unique()) == list(range(1, n_batches + 1)))
print("Duplicate event IDs (expect 0):", r["event_id"].duplicated().sum())
print("Replay IDs continue after the baseline:", r["event_id"].min() == base_max_id + 1)
bounds = r.groupby("batch_id")["event_ts"].agg(["min", "max"])
print("Every batch strictly later than the previous one:", bool((bounds["min"].iloc[1:].values > bounds["max"].iloc[:-1].values).all()))
print("Plan total matches the file:", plan["events_planned"].sum() == len(r))
print("Plan event types (expect 1064640 / 28683 / 9118):", plan[["views_planned", "carts_planned", "purchases_planned"]].sum().to_dict())

Replay rows (expect 1102441): 1102441
Batches: 9 | numbered 1..N without gaps: True
Duplicate event IDs (expect 0): 0
Replay IDs continue after the baseline: True
Every batch strictly later than the previous one: True
Plan total matches the file: True
Plan event types (expect 1064640 / 28683 / 9118): {'views_planned': 1064640, 'carts_planned': 28683, 'purchases_planned': 9118}
